# GRU (Gated Recurrent Unit) From Scratch

## Why do we need GRU?

A vanilla RNN has difficulty remembering information over long sequences because of the vanishing-gradient problem.

GRU solves this using **gates**.

Unlike LSTM:

- GRU has **no separate cell state**
- GRU has only one state: the hidden state $h_t$
- GRU has two gates:
  - **Update gate** $z_t$
  - **Reset gate** $r_t$
- GRU also creates a **candidate hidden state** $\tilde{h}_t$

The candidate state is the new information the GRU proposes to use.

The update gate decides how much of the candidate should replace the previous hidden state.

## GRU Cell

At every time step:

### 1. Update gate

$$
z_t = \sigma(W_z [h_{t-1},x_t] + b_z)
$$

It decides how much new information should be used.

### 2. Reset gate

$$
r_t = \sigma(W_r [h_{t-1},x_t] + b_r)
$$

It decides how much of the previous hidden state should be used when creating the candidate.

### 3. Candidate hidden state

$$
\tilde{h}_t =
\tanh(W_h[h_{t-1},x_t]_{\text{reset}} + b_h)
$$

where the hidden part is first multiplied by the reset gate:

$$
\tilde{h}_t =
\tanh(W_h[r_t \odot h_{t-1},x_t] + b_h)
$$

### 4. New hidden state

$$
h_t=(1-z_t)\odot h_{t-1}+z_t\odot\tilde{h}_t
$$

# Data

We use the same small **many-to-one sequence classification** style as the RNN and LSTM scratch implementations.

The sequence contains four time steps and each input has two features.

In [1]:
import numpy as np

In [2]:
# Many-to-one GRU

X = [
    [1.0, 0.5],
    [0.5, 1.0],
    [1.0, 1.5],
    [1.5, 1.0]
]

X = np.array(X)

# Class index
target = 1

X

array([[1. , 0.5],
       [0.5, 1. ],
       [1. , 1.5],
       [1.5, 1. ]])

# Parameters

In [3]:
input_size = 2
hidden_size = 2
output_size = 2

# Initial hidden state
h = np.zeros(hidden_size)

# Update Gate
Wz = np.array([
    [0.1, 0.2, 0.3, 0.4],
    [0.5, 0.6, 0.7, 0.8]
])
bz = np.array([0.1, 0.1])

# Reset Gate
Wr = np.array([
    [0.2, 0.1, 0.4, 0.3],
    [0.6, 0.5, 0.8, 0.7]
])
br = np.array([0.1, 0.1])

# Candidate Hidden State
Wh = np.array([
    [0.3, 0.2, 0.1, 0.4],
    [0.7, 0.6, 0.5, 0.8]
])
bh = np.array([0.0, 0.0])

# Output Layer
Why = np.array([
    [0.5, 0.3],
    [0.2, 0.4]
])
by = np.array([0.1, 0.2])

# Learning rate
lr = 0.01

# Activation Functions

In [4]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def tanh(x):
    return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

# GRU Forward Propagation

We store the values required later by BPTT:

- hidden states
- update gates
- reset gates
- candidate hidden states

In [5]:
def GRU_forward(X, h, Wz, bz, Wr, br, Wh, bh, Why, by):

    # Store states
    hidden_states = []

    update_gates = []
    reset_gates = []
    candidate_states = []

    # Initial hidden state
    hidden_states.append(h.copy())

    # Process sequence
    for t in range(len(X)):

        x_t = X[t]

        # Concatenate previous hidden state and current input
        z_input = np.concatenate([h, x_t])

        # Update Gate
        az = Wz @ z_input + bz
        z = sigmoid(az)

        # Reset Gate
        ar = Wr @ z_input + br
        r = sigmoid(ar)

        # Candidate Hidden State
        # Reset gate controls how much previous hidden
        # information is used to create the candidate.
        candidate_input = np.concatenate([r * h, x_t])

        ah = Wh @ candidate_input + bh
        candidate = tanh(ah)

        # New Hidden State
        h = (1 - z) * h + z * candidate

        # Store values
        hidden_states.append(h.copy())
        update_gates.append(z.copy())
        reset_gates.append(r.copy())
        candidate_states.append(candidate.copy())

    # Output Layer
    z_y = (Why @ h) + by

    def softmax(z):
        exp_z = np.exp(z - np.max(z))
        return exp_z / np.sum(exp_z)

    y = softmax(z_y)

    cache = {
        "hidden_states": hidden_states,
        "update_gates": update_gates,
        "reset_gates": reset_gates,
        "candidate_states": candidate_states
    }

    return y, cache

In [6]:
y, cache = GRU_forward(
    X, h,
    Wz, bz,
    Wr, br,
    Wh, bh,
    Why, by
)

print("Final prediction:", y)

Final prediction: [0.5029111 0.4970889]


# Loss Function

In [7]:
def CrossEntropyLoss(y, target):
    return -np.log(y[target])


loss = CrossEntropyLoss(y, target)

loss

np.float64(0.6989863949818254)

# BPTT (BackPropagation Through Time)

The output layer gives the gradient entering the final hidden state.

Then we move backward through every GRU time step.

For each time step, gradients flow through:

1. New hidden state
2. Update gate
3. Candidate state
4. Reset gate
5. Previous hidden state

We accumulate gradients for:

- $W_z, b_z$
- $W_r, b_r$
- $W_h, b_h$
- $W_{hy}, b_y$

In [8]:
def GRU_backward(
    X,
    y_true,
    cache,
    y_pred,
    Wz, bz,
    Wr, br,
    Wh, bh,
    Why, by,
    hidden_size
):

    # Final hidden state
    h_final = cache["hidden_states"][-1]

    # Softmax + Cross Entropy
    dz_y = y_pred - y_true

    # Output layer gradients
    dWhy = np.outer(dz_y, h_final)
    dby = dz_y.copy()

    # Gradient entering final hidden state
    dh_next = Why.T @ dz_y

    # Initialize gradients
    dWz = np.zeros_like(Wz)
    dbz = np.zeros_like(bz)

    dWr = np.zeros_like(Wr)
    dbr = np.zeros_like(br)

    dWh = np.zeros_like(Wh)
    dbh = np.zeros_like(bh)

    # BPTT
    for t in reversed(range(len(X))):

        # Stored values
        z = cache["update_gates"][t]
        r = cache["reset_gates"][t]
        candidate = cache["candidate_states"][t]

        h_prev = cache["hidden_states"][t]

        x_t = X[t]

        # Concatenated input used by update/reset gates
        z_input = np.concatenate([h_prev, x_t])

        # ------------------------------------------------
        # h_t = (1-z_t) * h_prev + z_t * candidate
        # ------------------------------------------------

        # Gradient through candidate
        d_candidate = dh_next * z

        # Gradient through update gate
        dz = dh_next * (candidate - h_prev)

        # Direct gradient to previous hidden state
        dh_prev = dh_next * (1 - z)

        # ------------------------------------------------
        # Candidate
        # candidate = tanh(Wh @ [r*h_prev, x_t] + bh)
        # ------------------------------------------------

        dah = d_candidate * (1 - candidate ** 2)

        candidate_input = np.concatenate([r * h_prev, x_t])

        dWh += np.outer(dah, candidate_input)
        dbh += dah

        # Gradient entering candidate input
        dcandidate_input = Wh.T @ dah

        # Hidden part was r * h_prev
        d_rh_prev = dcandidate_input[:hidden_size]

        # Input part is not needed for parameter updates
        # because X is fixed training data.

        # Gradient through r * h_prev
        dr = d_rh_prev * h_prev
        dh_prev += d_rh_prev * r

        # ------------------------------------------------
        # Reset gate
        # ------------------------------------------------

        dar = dr * r * (1 - r)

        dWr += np.outer(dar, z_input)
        dbr += dar

        # Gradient entering [h_prev, x_t]
        dreset_input = Wr.T @ dar
        dh_prev += dreset_input[:hidden_size]

        # ------------------------------------------------
        # Update gate
        # ------------------------------------------------

        daz = dz * z * (1 - z)

        dWz += np.outer(daz, z_input)
        dbz += daz

        # Gradient entering [h_prev, x_t]
        dupdate_input = Wz.T @ daz
        dh_prev += dupdate_input[:hidden_size]

        # Pass gradient to previous time step
        dh_next = dh_prev

    gradients = {
        "Wz": dWz,
        "bz": dbz,

        "Wr": dWr,
        "br": dbr,

        "Wh": dWh,
        "bh": dbh,

        "Why": dWhy,
        "by": dby
    }

    return gradients

In [9]:
one_hot_target = np.zeros(output_size)
one_hot_target[target] = 1

gradients = GRU_backward(
    X,
    one_hot_target,
    cache,
    y,
    Wz, bz,
    Wr, br,
    Wh, bh,
    Why, by,
    hidden_size
)

gradients

{'Wz': array([[2.16024807e-03, 4.08414812e-03, 5.59307388e-03, 6.67949803e-03],
        [4.73601183e-05, 1.40559258e-04, 4.85991783e-04, 4.57002836e-04]]),
 'bz': array([0.0055439 , 0.00060167]),
 'Wr': array([[0.00145364, 0.00225097, 0.00330992, 0.00262247],
        [0.00045334, 0.00074823, 0.0010449 , 0.00095556]]),
 'br': array([0.00242101, 0.00086293]),
 'Wh': array([[ 0.03584255,  0.06959931,  0.1110587 ,  0.09243857],
        [-0.00052007, -0.00069113, -0.00032903,  0.00034839]]),
 'bh': array([0.08616441, 0.00075045]),
 'Why': array([[ 0.35227923,  0.49536497],
        [-0.35227923, -0.49536497]]),
 'by': array([ 0.5029111, -0.5029111])}

# Optimization

In [10]:
def optimization(
    Wz, bz,
    Wr, br,
    Wh, bh,
    Why, by,
    gradients
):

    Wz = Wz - (lr * gradients["Wz"])
    bz = bz - (lr * gradients["bz"])

    Wr = Wr - (lr * gradients["Wr"])
    br = br - (lr * gradients["br"])

    Wh = Wh - (lr * gradients["Wh"])
    bh = bh - (lr * gradients["bh"])

    Why = Why - (lr * gradients["Why"])
    by = by - (lr * gradients["by"])

    return (
        Wz, bz,
        Wr, br,
        Wh, bh,
        Why, by
    )

# Training

In [11]:
epochs = 1000

for _ in range(epochs):

    # Forward pass
    y, cache = GRU_forward(
        X, h,
        Wz, bz,
        Wr, br,
        Wh, bh,
        Why, by
    )

    # Loss
    loss = CrossEntropyLoss(y, target)

    # Calculate gradients
    gradients = GRU_backward(
        X,
        one_hot_target,
        cache,
        y,
        Wz, bz,
        Wr, br,
        Wh, bh,
        Why, by,
        hidden_size
    )

    # Optimize
    (
        Wz, bz,
        Wr, br,
        Wh, bh,
        Why, by
    ) = optimization(
        Wz, bz,
        Wr, br,
        Wh, bh,
        Why, by,
        gradients
    )

    if _ % 100 == 0:
        print(f"Epoch {_} : loss {loss} : Prediction {y}")

Epoch 0 : loss 0.6989863949818254 : Prediction [0.5029111 0.4970889]
Epoch 100 : loss 0.20672608139086698 : Prediction [0.18675762 0.81324238]
Epoch 200 : loss 0.10962314896019194 : Prediction [0.10382821 0.89617179]
Epoch 300 : loss 0.0720547810947095 : Prediction [0.06952008 0.93047992]
Epoch 400 : loss 0.05287583512873626 : Prediction [0.05150222 0.94849778]
Epoch 500 : loss 0.041455152507362626 : Prediction [0.04060764 0.95939236]
Epoch 600 : loss 0.03395287727317115 : Prediction [0.03338295 0.96661705]
Epoch 700 : loss 0.028678848602941266 : Prediction [0.02827151 0.97172849]
Epoch 800 : loss 0.024783192252272393 : Prediction [0.02447861 0.97552139]
Epoch 900 : loss 0.021795405127367024 : Prediction [0.0215596 0.9784404]


# Final GRU Flow

The complete scratch implementation is:

$$
x_t, h_{t-1}
$$

↓

**Update Gate**

$$
z_t = \sigma(W_z[h_{t-1},x_t]+b_z)
$$

↓

**Reset Gate**

$$
r_t = \sigma(W_r[h_{t-1},x_t]+b_r)
$$

↓

**Candidate**

$$
\tilde h_t =
\tanh(W_h[r_t\odot h_{t-1},x_t]+b_h)
$$

↓

**New Hidden State**

$$
h_t=(1-z_t)h_{t-1}+z_t\tilde h_t
$$

↓

Repeat for every time step

↓

**Softmax → Cross Entropy → BPTT → Gradient Descent**

This is the complete GRU training loop from scratch.